<a href="https://colab.research.google.com/github/MuhammedHarisAliKhan/AI-Based-Chatbot/blob/main/notebooks/Getting_started_with_google_colab_ai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import sqlite3
import nltk
import gradio as gr
import string

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

nltk.download('punkt')
nltk.download('stopwords')

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [12]:
conn = sqlite3.connect("chatbot.db")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS faqs (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    question TEXT,
    answer TEXT
)
""")

# 🔥 HUGE FAQ DATASET
faqs = [
    ("how to login", "Enter your email and password."),
    ("how to sign up", "Click sign up and fill form."),
    ("how to reset password", "Click forgot password link."),
    ("how to delete account", "Go to settings and delete account."),
    ("how to change email", "Go to profile settings."),
    ("how to update profile", "Edit profile section."),
    ("how to contact support", "Use help section."),
    ("how to track order", "Go to orders page."),
    ("how to cancel order", "Click cancel button in orders."),
    ("how to refund", "Refund is processed in 5-7 days."),
    ("payment failed", "Try again or use another card."),
    ("how to add address", "Go to profile and add address."),
    ("app not working", "Restart app or clear cache."),
    ("how to change password", "Go to settings and update password."),
    ("how to logout", "Click logout button."),

    ("how to login", "Enter your email and password to login."),
    ("how to sign up", "Click on sign up and fill the registration form."),
    ("how to reset password", "Click on forgot password option and follow steps."),
    ("how to delete account", "Go to settings and click delete account."),
    ("how to change password", "Go to settings and update your password."),
    ("how to logout", "Click on logout button from menu."),
    ("app not working", "Restart the app or clear cache."),
    ("payment failed", "Try another card or payment method."),
    ("how to track order", "Go to orders section and track your order."),
    ("how to cancel order", "Open order and click cancel."),
    ("how to refund", "Refund is processed in 5-7 working days."),
    ("how to add address", "Go to profile and add new address."),
    ("how to update profile", "Edit your profile section."),
    ("how to contact support", "Use help or support section."),
    ("how to change email", "Go to account settings and update email."),


    ("forgot username", "Check your email for username recovery."),
    ("account locked", "Contact support to unlock your account."),
    ("two factor authentication", "Enable 2FA from security settings."),
    ("app crashing", "Reinstall the app or update it."),
    ("order not received", "Contact support with order ID."),
    ("wrong password error", "Reset your password using forgot option."),
    ("how to deactivate account", "Go to settings and deactivate account."),
    ("how to update payment method", "Go to billing section and update card."),
    ("subscription cancel", "Go to subscription and cancel plan."),
    ("how to upgrade plan", "Select premium plan from pricing page."),


    ("delivery time", "Delivery takes 3-5 business days."),
    ("shipping charges", "Shipping is free above $50."),
    ("international shipping", "Yes, we ship worldwide."),
    ("product return policy", "Returns accepted within 14 days."),
    ("order confirmation", "You will receive email confirmation."),
    ("discount code not working", "Check expiry date of coupon."),
    ("how to apply coupon", "Enter coupon at checkout page."),
    ("account verification", "Verify email using link sent."),
]
cursor.executemany("INSERT INTO faqs (question, answer) VALUES (?, ?)", faqs)
conn.commit()

In [13]:
def preprocess(text):
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    tokens = word_tokenize(text)

    stop_words = set(stopwords.words('english'))
    return " ".join([w for w in tokens if w not in stop_words])

In [14]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
cursor.execute("SELECT question, answer FROM faqs")
data = cursor.fetchall()

questions = [row[0] for row in data]
answers = [row[1] for row in data]

processed_questions = [preprocess(q) for q in questions]

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [8]:
cursor.execute("SELECT question, answer FROM faqs")
data = cursor.fetchall()

questions = [row[0] for row in data]
answers = [row[1] for row in data]

processed_questions = [preprocess(q) for q in questions]

In [9]:
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(processed_questions)

In [10]:
def chatbot(user_input):
    user_input = preprocess(user_input)
    user_vec = vectorizer.transform([user_input])

    similarity = cosine_similarity(user_vec, X)
    index = similarity.argmax()

    if similarity[0][index] < 0.2:
        return "Sorry, I don't understand your question."

    return answers[index]

In [15]:
interface = gr.Interface(
    fn=chatbot,
    inputs="text",
    outputs="text",
    title="🔥 AI FAQ Chatbot",
    description="Ask anything like login, signup, payment, support etc."
)

interface.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://71c6709672877229fe.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
